#3 - A partir das extremidades das águas correntes e das águas correntes estimadas, traçar um círculo de 50m;

Desculpa a minha demora aqui... eu tive um problema com meu convênio de saúde e fiquei birutinha. OK, vamos tentar voltar pro corpo, né?  
  
Acho que teríamos que fazer:
* Comparar os sem identificação (SD) entre o drenagem e o correntes estimadas
* Aquele processo do Henrique tanto no drenagem (que ele já fez, mas quando for pro outro github eu vou dar uma mudada)... (? quanto no de correntes estimadas do Elias)
* Pegar os pontinhos no drenaghenrique e fazer um simmetrical difference com os pontos do correntes_estimadas...
* ... aí eu preciso conferir, né (ainda n sei como)

In [1]:
import geopandas as gpd
from os.path import join
from tqdm import tqdm

In [2]:
def testar_gdf(gdf):
    print(
    f'Shape: {gdf.shape};\n'+
    f'\nSample:{gdf.sample()}'
)

Ok, vamos começar do começo!
# A. Comparar drenagem ([GeoSampa](http://wfs.geosampa.prefeitura.sp.gov.br/geoserver/geoportal/wfs?version=1.0.0&request=GetFeature&outputFormat=SHAPE-ZIP&typeName=geoportal:drenagem)) e correntes estimadas ([Elias](https://github.com/sepep-pmsp/siiau/blob/master/projects/urbis/assets/silver/parquet_aguas_correntes_estimadas.py))

In [3]:
drenageo = gpd.read_file(
    join(
        'data',
        'drenagem.zip'
    )
)
print(
    f'Shape: {drenageo.shape};\n'+
    f'\nSample:{drenageo.sample()}'
)

## Como as correntes estimadas tem a id como int, vou transformar a id do drenageo em int tbm
drenageo['cd_identif'].astype('int', copy=False)

Shape: (27611, 15);

Sample:      cd_identif cd_tipo_ac tx_tipo_ac cd_numero_ nm_bairro nm_acident  \
4135      3893.0         ND       None          1       S/B         SD   

      qt_comprim  cd_tipo_cu                nm_tipo_cu nm_via_pro nm_descrit  \
4135   45.766966        11.0  Trecho em estado natural        S/N       None   

               nm_tipo_tr dt_atualiz cd_usuario  \
4135  Trecho a céu aberto 2025-01-03       None   

                                               geometry  
4135  LINESTRING (327073.124 7363048.935, 327071.319...  


0            1
1        26125
2            2
3        26126
4        26127
         ...  
27606    27596
27607    27597
27608    27598
27609    27599
27610    27600
Name: cd_identif, Length: 27611, dtype: int64

In [4]:
##... Mas antes disso, vamos ver se este parquet tá realmente dando explore e afins
correntes_estimadas = gpd.read_parquet(
    join(
        'data',
        'aguas_correntes_estimadas.parquet'
    )
)
print(
    f'Shape: {correntes_estimadas.shape};\n'+
    f'\nSample:{correntes_estimadas.sample()}'
)

Shape: (19446, 15);

Sample:                                               geometry  cd_identificador  \
9692  POLYGON ((328814.194 7349797.571, 328824.002 7...             13388   

     cd_tipo_acidente tx_tipo_acidente cd_numero_ordem nm_bairro nm_acidente  \
9692               ND                                1      None          SD   

      qt_comprimento_hidrografia  cd_tipo_curso_hidrografia  \
9692                  458.061135                         11   

     nm_tipo_curso_hidrografia nm_via_proximo nm_descritivo  \
9692  Trecho em estado natural           None          None   

           nm_tipo_trecho        dt_atualizacao cd_usuario_atualizacao  
9692  Trecho a céu aberto  2025-01-03T03:00:00Z                   None  


In [5]:
drenageo_in_estim=drenageo.loc[drenageo['cd_identif'].isin(correntes_estimadas['cd_identificador'].astype(dtype='float'))]
testar_gdf(drenageo_in_estim)


Shape: (19446, 15);

Sample:      cd_identif cd_tipo_ac tx_tipo_ac cd_numero_ nm_bairro nm_acident  \
7863      7439.0         ND       None          1       S/B         SD   

      qt_comprim  cd_tipo_cu                nm_tipo_cu  \
7863  410.350443        11.0  Trecho em estado natural   

                      nm_via_pro nm_descrit           nm_tipo_tr dt_atualiz  \
7863  DA VARGEM GRANDE A COLONIA       None  Trecho a céu aberto 2025-01-03   

     cd_usuario                                           geometry  
7863       None  LINESTRING (324926.524 7359856.086, 324928.197...  


Provavelmente eles são iguais mesmo, só que as correntes estimadas são polígonos ao invés de linhas  
# B. Conferir SDs
Vamos conferir quais os 'nm_acidentes' que aparecem e se tem SD ou outra forma de dados vazios nesse recorte